## 01_scRNA-seq Cancer Classification

### Objective

This notebook presents a complete machine learning pipeline for **cancer type classification** using single-cell RNA sequencing (scRNA-seq) data.

The goal is to build and evaluate models capable of distinguishing between multiple cancer types based on gene expression profiles.

---

### Pipeline Overview

The analysis follows a structured workflow aligned with best practices in biomedical machine learning:

1. **Data exploration**
   - Dataset inspection and class distribution analysis

2. **Preprocessing**
   - Log transformation of gene expression values
   - Selection of Highly Variable Genes (HVG)
   - Feature scaling

3. **Dimensionality reduction**
   - Principal Component Analysis (PCA)
   - Selection of components based on explained variance

4. **Model training**
   - Multiple algorithms: Logistic Regression, Random Forest, SVM, Gradient Boosting

5. **Evaluation**
   - Accuracy, F1-score (macro and weighted)
   - Confusion matrix analysis

6. **Model validation**
   - Stratified train-test split
   - Cross-validation for robustness

7. **Hyperparameter tuning**
   - Grid Search and Random Search optimization

8. **Deep learning approach**
   - Neural network for comparison with classical models

---

### Key Focus

This notebook emphasizes:

- Handling **high-dimensional biological data**
- Preventing **overfitting**
- Ensuring **robust and fair evaluation**
- Comparing multiple models under the **No Free Lunch principle**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Load data
X = pd.read_csv("data/scRNA_data.csv", index_col=0)
y = pd.read_csv("data/sc_RNA_labels.csv", index_col=0)

print("✓ Raw dimensions:", X.shape)
print("✓ Label shape:", y.shape)

## 🧬 Input Data: scRNA-seq Dataset

### Overview

This analysis uses single-cell RNA sequencing (scRNA-seq) data from **five cancer types**, encompassing:

| Metric | Value |
|--------|-------|
| **Total Samples** | 801 cells |
| **Gene Features** | 20,531 genes |
| **Missing Values** | 0 (complete dataset) |

### Challenges in High-Dimensional Biological Data

The dataset exhibits a challenging structure inherent to scRNA-seq analyses:

- **Features >> Samples** (20,531 features vs 801 samples)
- **Curse of dimensionality** - increased overfitting risk
- **High noise levels** - technical and biological variability
- **Feature redundancy** - many genes encode correlated information

These characteristics necessitate **preprocessing, feature selection, and dimensionality reduction** before machine learning model training.

In [ ]:
print("Features:", X.shape[1])
print("Samples:", X.shape[0])

print("\nMissing values:")
print(X.isnull().sum().sum())

print("\nClass distribution:")
print(y.value_counts())

In [ ]:
print(y['Class'].value_counts())

## 📊 Class Distribution: Multi-Cancer Classification

### Sample Distribution by Cancer Type

| Cancer Type | Sample Count | Percentage |
|-------------|-------------|-----------|
| **BRCA** (Breast Carcinoma) | 300 | 37.5% |
| **KIRC** (Kidney Renal Clear Cell) | 146 | 18.2% |
| **LUAD** (Lung Adenocarcinoma) | 141 | 17.6% |
| **PRAD** (Prostate Adenocarcinoma) | 136 | 17.0% |
| **COAD** (Colorectal Adenocarcinoma) | 78 | 9.7% |

### Implications for Model Training

⚠️ **Class Imbalance**: The dataset exhibits significant **imbalance**, with BRCA representing 37.5% and COAD only 9.7% of samples.

**Recommended evaluation metrics**:
- ✓ Macro F1-score (unweighted average across classes)
- ✓ Weighted F1-score (accounts for class prevalence)
- ✓ Confusion matrix (per-class performance analysis)

In [ ]:
# Step 1: Log transformation (no leakage — element-wise, no fitting needed)
X_log = np.log1p(X)
y_classes = y['Class'].values

print(f"✓ After log1p transformation: {X_log.shape}")

## 🔧 Feature Preprocessing Pipeline

### ⚠️ Important: Order of Operations to Avoid Data Leakage

The correct order is:
1. Log transformation (safe — no fitting, purely element-wise)
2. **Train-test split FIRST**
3. HVG selection computed only on train
4. StandardScaler fit only on train
5. PCA fit only on train

This ensures the test set remains completely unseen during all fitting steps.

In [ ]:
# Step 2: Stratified train-test split — BEFORE any fitting
# ✅ CORRECTED: split happens before HVG selection, scaling, and PCA
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_log, y_classes,
    test_size=0.2,
    stratify=y_classes,
    random_state=42
)

print(f"✓ Train-Test Split (Stratified):")
print(f"  Train set: {X_train_raw.shape[0]} samples")
print(f"  Test set:  {X_test_raw.shape[0]} samples")
print(f"  Train class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")
print(f"\n  Test class distribution:\n{pd.Series(y_test).value_counts().sort_index()}")

## ⚙️ Stratified Train-Test Split

The dataset was split into training (80%) and testing (20%) sets using **stratified sampling**.

### Why Split FIRST?

Any fitting step (HVG selection, StandardScaler, PCA) must happen **after** the split,
using **only training data**. Otherwise, the model indirectly learns from the test set
before evaluation — this is called **data leakage**.

With stratification:
- ✓ Both sets maintain ~37.5% BRCA, ~18% KIRC, etc.
- ✓ Unbiased performance assessment across all cancer types
- ✓ Honest estimate of real-world generalization ability

In [ ]:
# Step 3: HVG selection — computed ONLY on train
# ✅ CORRECTED: gene variance computed on X_train_raw only
gene_vars_train = X_train_raw.var(axis=0).sort_values(ascending=False)
n_hvg = 2000
hvg_genes = gene_vars_train.head(n_hvg).index

X_train_hvg = X_train_raw[hvg_genes]
X_test_hvg  = X_test_raw[hvg_genes]   # same genes selected, no refitting

print(f"✓ After HVG selection (top {n_hvg} genes, computed on TRAIN only):")
print(f"  Train: {X_train_hvg.shape}")
print(f"  Test:  {X_test_hvg.shape}")
print(f"  Gene variance range (train): {gene_vars_train.min():.4f} - {gene_vars_train.max():.4f}")

In [ ]:
# Step 4: StandardScaler — fit ONLY on train, transform both
# ✅ CORRECTED: scaler.fit_transform on train, scaler.transform on test
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_hvg)
X_test_scaled  = scaler.transform(X_test_hvg)

print(f"✓ After StandardScaler (fit on TRAIN only):")
print(f"  Train — mean: {X_train_scaled.mean():.6f}, std: {X_train_scaled.std():.6f}")
print(f"  Test  — mean: {X_test_scaled.mean():.6f},  std: {X_test_scaled.std():.6f}")

## 📉 Dimensionality Reduction: Principal Component Analysis

### Why PCA is Essential

Even after HVG selection (2,000 genes), the feature space remains **high-dimensional**
relative to sample size (801 samples).

### ✅ Corrected Approach

PCA is **fit only on training data**. The test set is projected using the axes
learned from train. This prevents the principal components from being influenced
by test set variance.

In [ ]:
# Step 5: PCA — fit ONLY on train, transform both
# ✅ CORRECTED: pca.fit on train only
pca_full = PCA()
pca_full.fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

n_components_90 = np.argmax(cum_var >= 0.90) + 1
n_components_95 = np.argmax(cum_var >= 0.95) + 1

pca = PCA(n_components=n_components_90)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"✓ PCA Analysis (fit on TRAIN only):")
print(f"  Components for 90% variance: {n_components_90}")
print(f"  Components for 95% variance: {n_components_95}")
print(f"  Selected: {pca.n_components_} components")
print(f"  Variance retained: {pca.explained_variance_ratio_.sum():.4f}")
print(f"  Train final shape: {X_train_pca.shape}")
print(f"  Test  final shape: {X_test_pca.shape}")

# Visualize scree plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(np.cumsum(pca_full.explained_variance_ratio_[:50]))
plt.axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
plt.xlabel('Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Scree Plot (fit on train)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.bar(range(min(20, len(pca.explained_variance_ratio_))),
        pca.explained_variance_ratio_[:20])
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance by Component')
plt.tight_layout()
plt.show()

## 📊 PCA Results & Interpretation

### Dimensionality Reduction Summary

| Stage | Features | Samples | Variance Retained |
|-------|----------|---------|-------------------|
| **Original** | 20,531 | 801 | 100% |
| **After HVG Selection** | 2,000 | 801 | 100% (subset) |
| **After PCA (90% threshold)** | ~300 components | 801 | **~90%** |

We selected **the 90% threshold** because:
1. **Biological Relevance** - Retains sufficient signal to distinguish cancer types
2. **Statistical Stability** - Avoids overfitting to minor variance components
3. **Computational Efficiency** - Dramatic reduction from 2,000 → ~300 features
4. **Standard Practice** - 90% threshold is convention in medical ML applications

In [ ]:
# Step 6: Train models on leak-free data
models = {
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM (RBF)':           SVC(kernel='rbf', random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42)
}

results = {}
print("✓ Model Performance (on TEST set):\n")
print(f"{'Model':<20} {'Accuracy':<12} {'F1-Score (macro)':<20} {'F1-Score (weighted)':<20}")
print("-" * 72)

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)

    accuracy    = (y_pred == y_test).mean()
    f1_macro    = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    results[name] = {'accuracy': accuracy, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}
    print(f"{name:<20} {accuracy:<12.4f} {f1_macro:<20.4f} {f1_weighted:<20.4f}")

best_model_name = max(results, key=lambda x: results[x]['f1_weighted'])
print(f"\n✓ Best Model: {best_model_name} (by weighted F1-score)")
print(f"  Classification Report:\n")
print(classification_report(y_test, models[best_model_name].predict(X_test_pca)))

## 🤖 Classifier Ensemble: Multi-Algorithm Comparison

### Model Selection Strategy

We train and compare **four diverse algorithms** representing different learning paradigms:

| Algorithm | Type | Characteristics |
|-----------|------|-----------------|
| **Logistic Regression** | Linear | Interpretable, fast, baseline model |
| **Random Forest** | Tree-based Ensemble | Robust, handles non-linearity, feature importances |
| **Support Vector Machine (SVM)** | Kernel-based | Excellent in high dimensions, robust |
| **Gradient Boosting** | Sequential Ensemble | Often highest accuracy, captures complex patterns |

### The No Free Lunch Theorem

**Key Principle**: There is **no universally superior algorithm** that outperforms all others on all datasets.
We must **empirically test multiple algorithms** and compare performance metrics.

### Evaluation Metrics

- **Accuracy**: Overall correctness (⚠️ misleading for imbalanced data)
- **Macro F1-Score**: Unweighted average across all cancer types
- **Weighted F1-Score**: Accounts for class prevalence
- **Confusion Matrix**: Per-cancer-type breakdown

In [ ]:
# Step 7: Confusion Matrix Analysis
best_model  = models[best_model_name]
y_pred_best = best_model.predict(X_test_pca)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y_classes),
            yticklabels=np.unique(y_classes))
plt.title(f'Confusion Matrix — {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print(f"✓ Confusion Matrix Analysis:")
print(f"  Diagonal elements (correct classifications): {np.diag(cm)}")
print(f"  Total test samples: {cm.sum()}")
print(f"  Correct predictions: {np.diag(cm).sum()}")
print(f"  Misclassifications: {cm.sum() - np.diag(cm).sum()}")

## ⚠️ Critical Interpretation: High Performance in Biological Data

The models achieve **exceptional performance** (>98% accuracy, near-perfect F1-scores).

### Potential Explanations

1. **Strong Biological Signal** ✓ — Cancer types may have genuinely distinct transcriptomic signatures
2. **Dataset Characteristics** ⚠️ — Batch effects or technical artifacts may separate classes

### Best Practices for Validation

- **External validation**: Test on independent patient cohorts
- **Batch effect analysis**: Control for technical confounders
- **Interpretability**: Examine which genes drive predictions

In [ ]:
# Step 8: Cross-Validation with Pipeline
# ✅ CORRECTED: Pipeline ensures scaler+PCA refit on each fold's train data only
print("━" * 70)
print("✓ CROSS-VALIDATION with Pipeline (no leakage between folds)")
print("━" * 70)

best_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=n_components_90)),
    ('clf',    models[best_model_name].__class__(
                   **models[best_model_name].get_params()
               ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ✅ Input is X_train_hvg (post-HVG, pre-scaling) — pipeline handles the rest
cv_scores = cross_val_score(
    best_pipeline,
    X_train_hvg,
    y_train,
    cv=cv,
    scoring='f1_weighted'
)

print(f"\n✓ 5-Fold CV Results ({best_model_name} inside Pipeline):")
print(f"  Fold scores: {cv_scores.round(4)}")
print(f"  Mean CV F1 : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"  Test set F1: {results[best_model_name]['f1_weighted']:.4f}")

if cv_scores.std() > 0.05:
    print("\n  ⚠️ High variance in CV — model may be unstable")
else:
    print("\n  ✓ Low variance in CV — model is stable across folds")

## 📋 Cross-Validation Results & Robustness Analysis

### ✅ Why Pipeline is Essential for CV

Without a Pipeline, `cross_val_score` only cross-validates the classifier.
The StandardScaler and PCA were already fit on all training data — including
what becomes the validation fold in each iteration. This is **leakage inside CV**.

With a Pipeline, in each fold:
- Scaler is fit on the 4 training folds only
- PCA is fit on the 4 training folds only
- The held-out fold is transformed (not fit) — truly unseen

In [ ]:
# Step 9: Hyperparameter Tuning — GridSearch & RandomSearch on Pipelines
# ✅ CORRECTED: search is over Pipeline objects, not bare classifiers
print("=" * 80)
print("HYPERPARAMETER TUNING — with Pipeline (no leakage)")
print("=" * 80)

# 1️⃣ Grid Search: SVM
print("\n1️⃣ GRID SEARCH: SVM\n")
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=n_components_90)),
    ('clf',    SVC())
])
param_grid_svm = {
    'clf__C':      [0.1, 1, 10, 100],
    'clf__kernel': ['linear', 'rbf'],
    'clf__gamma':  ['scale', 'auto']
}
grid_svm = GridSearchCV(pipe_svm, param_grid_svm, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_svm.fit(X_train_hvg, y_train)
y_pred_svm_tuned = grid_svm.predict(X_test_hvg)
f1_svm_tuned = f1_score(y_test, y_pred_svm_tuned, average='weighted')
print(f"\n   Best parameters : {grid_svm.best_params_}")
print(f"   Best CV score   : {grid_svm.best_score_:.4f}")
print(f"   Test F1-score   : {f1_svm_tuned:.4f}")
print(f"   vs baseline SVM : {f1_svm_tuned - results['SVM (RBF)']['f1_weighted']:+.4f}")

# 2️⃣ Grid Search: Logistic Regression
print("\n" + "─" * 80)
print("2️⃣ GRID SEARCH: Logistic Regression\n")
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=n_components_90)),
    ('clf',    LogisticRegression())
])
param_grid_lr = {
    'clf__C':        [0.001, 0.01, 0.1, 1, 10],
    'clf__solver':   ['lbfgs', 'liblinear'],
    'clf__max_iter': [1000, 2000]
}
grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_lr.fit(X_train_hvg, y_train)
y_pred_lr_tuned = grid_lr.predict(X_test_hvg)
f1_lr_tuned = f1_score(y_test, y_pred_lr_tuned, average='weighted')
print(f"\n   Best parameters : {grid_lr.best_params_}")
print(f"   Best CV score   : {grid_lr.best_score_:.4f}")
print(f"   Test F1-score   : {f1_lr_tuned:.4f}")
print(f"   vs baseline LR  : {f1_lr_tuned - results['Logistic Regression']['f1_weighted']:+.4f}")

# 3️⃣ Random Search: Random Forest
print("\n" + "─" * 80)
print("3️⃣ RANDOM SEARCH: Random Forest\n")
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=n_components_90)),
    ('clf',    RandomForestClassifier(random_state=42))
])
param_dist_rf = {
    'clf__n_estimators':      [50, 100, 200, 300],
    'clf__max_depth':         [10, 20, 30, None],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf':  [1, 2, 4],
    'clf__max_features':      ['sqrt', 'log2']
}
random_rf = RandomizedSearchCV(pipe_rf, param_dist_rf, n_iter=20, cv=5,
                                scoring='f1_weighted', n_jobs=-1, verbose=1, random_state=42)
random_rf.fit(X_train_hvg, y_train)
y_pred_rf_tuned = random_rf.predict(X_test_hvg)
f1_rf_tuned = f1_score(y_test, y_pred_rf_tuned, average='weighted')
print(f"\n   Best parameters : {random_rf.best_params_}")
print(f"   Best CV score   : {random_rf.best_score_:.4f}")
print(f"   Test F1-score   : {f1_rf_tuned:.4f}")
print(f"   vs baseline RF  : {f1_rf_tuned - results['Random Forest']['f1_weighted']:+.4f}")

# Summary
print("\n" + "=" * 80)
print("HYPERPARAMETER TUNING SUMMARY")
print("=" * 80)
tuning_results = {
    'SVM (Tuned)':                 f1_svm_tuned,
    'Logistic Regression (Tuned)': f1_lr_tuned,
    'Random Forest (Tuned)':       f1_rf_tuned,
}
print(f"\n{'Model':<30} {'F1 Baseline':<15} {'F1 Tuned':<15} {'Δ':<10}")
print("-" * 70)
for label, f1_tuned, base_key in [
    ('SVM',                 f1_svm_tuned, 'SVM (RBF)'),
    ('Logistic Regression', f1_lr_tuned,  'Logistic Regression'),
    ('Random Forest',       f1_rf_tuned,  'Random Forest'),
]:
    baseline = results[base_key]['f1_weighted']
    print(f"{label:<30} {baseline:<15.4f} {f1_tuned:<15.4f} {f1_tuned - baseline:+.4f}")

best_tuned_model = max(tuning_results, key=tuning_results.get)
print(f"\n✓ Best tuned model: {best_tuned_model} ({tuning_results[best_tuned_model]:.4f})")

## 🔧 Hyperparameter Tuning & Optimization

### ✅ Why Pipeline is Essential for GridSearch

When `GridSearchCV` performs its internal cross-validation to score each parameter
combination, it must re-fit all preprocessing steps on each fold's training data.
Using a Pipeline guarantees this automatically.

Parameter names use the `clf__` prefix to refer to the classifier inside the pipeline
(e.g., `clf__C` instead of just `C`).

| Approach | Method | Models | Advantage |
|----------|--------|--------|-----------|
| **Grid Search** | Exhaustive search | SVM, Logistic Regression | Guarantees best in grid |
| **Random Search** | Random sampling | Random Forest | Efficient for large spaces |

In [ ]:
# Step 10: Neural Network
# ✅ Uses X_train_pca / X_test_pca which are already leak-free
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("=" * 80)
print("NEURAL NETWORK TRAINING")
print("=" * 80)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded  = le.transform(y_test)
n_classes = len(le.classes_)

print(f"\n✓ Label encoding: {dict(zip(le.classes_, range(n_classes)))}")

model_nn = keras.Sequential([
    layers.Input(shape=(X_train_pca.shape[1],)),
    layers.Dense(128, activation='relu', name='dense_1'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', name='dense_2'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu', name='dense_3'),
    layers.Dropout(0.2),
    layers.Dense(n_classes, activation='softmax', name='output')
])

model_nn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model Architecture:")
model_nn.summary()

print("\n✓ Training Neural Network...")
history = model_nn.fit(
    X_train_pca, y_train_encoded,
    validation_data=(X_test_pca, y_test_encoded),
    epochs=100,
    batch_size=32,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    ]
)

y_pred_nn         = model_nn.predict(X_test_pca, verbose=0).argmax(axis=1)
y_pred_nn_classes = le.inverse_transform(y_pred_nn)
accuracy_nn       = (y_pred_nn == y_test_encoded).mean()
f1_macro_nn       = f1_score(y_test, y_pred_nn_classes, average='macro')
f1_weighted_nn    = f1_score(y_test, y_pred_nn_classes, average='weighted')

print(f"\n✓ Neural Network Performance (Test Set):")
print(f"  Accuracy:            {accuracy_nn:.4f}")
print(f"  F1-Score (macro):    {f1_macro_nn:.4f}")
print(f"  F1-Score (weighted): {f1_weighted_nn:.4f}")
print(f"  Epochs trained:      {len(history.history['loss'])}")
print(f"  Final train loss:    {history.history['loss'][-1]:.4f}")
print(f"  Final val loss:      {history.history['val_loss'][-1]:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Neural Network: Loss over Epochs')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('Neural Network: Accuracy over Epochs')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Neural Network training complete!")

## 🧠 Neural Network Classifier

### Architecture

| Layer | Configuration | Purpose |
|-------|---------------|---------|
| Input | ~300 features | PCA-transformed data |
| Dense #1 | 128 neurons + ReLU + BN | Feature learning |
| Dropout | 0.3 | Prevent overfitting |
| Dense #2 | 64 neurons + ReLU + BN | Higher-level features |
| Dropout | 0.3 | Prevent overfitting |
| Dense #3 | 32 neurons + ReLU | Compression |
| Dropout | 0.2 | Prevent overfitting |
| Output | 5 neurons + Softmax | 5-class probability |

In [ ]:
# Final model comparison chart
model_names = ['Gradient\nBoosting', 'Random\nForest\n(tuned)', 'SVM\n(tuned)', 'Neural\nNetwork', 'Logistic Reg\n(tuned)']
f1_scores_plot = [
    results['Gradient Boosting']['f1_weighted'],
    f1_rf_tuned,
    f1_svm_tuned,
    f1_weighted_nn,
    f1_lr_tuned
]
colors = ['#64748B', '#1E7DB0', '#1E7DB0', '#0D9488', '#0D9488']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(model_names, f1_scores_plot, color=colors, width=0.6)
for bar, val in zip(bars, f1_scores_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0.94, 1.01)
ax.set_ylabel('Weighted F1-Score')
ax.set_title('Model Comparison — Weighted F1-Score (Test Set)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("model_comparison_f1.png", dpi=150, bbox_inches='tight')
plt.show()

## 📊 Comprehensive Model Comparison & Strategic Analysis

### All Models: Baseline vs Tuned vs Neural Network

| Model | F1 (Baseline) | F1 (Tuned) | Type | Strengths | Weaknesses |
|-------|---------------|-----------|------|-----------|-----------|
| **Neural Network** | N/A | **~0.99** | Deep Learning | Flexible, non-linear | Requires more data, black-box |
| **Logistic Regression (Tuned)** | ~0.99 | **~0.99** | Linear | Fast, interpretable | Limited non-linearity |
| **SVM (Tuned)** | ~0.98 | **~0.99** | Kernel | High-dim robust | Slow on large data |
| **Random Forest (Tuned)** | ~0.98 | **~0.99** | Ensemble | Feature importances | Can overfit |
| **Gradient Boosting** | ~0.97 | — | Sequential | Strong learner | Slowest, overfit risk |

### No Free Lunch Theorem Verified ✓

| Principle | Observation |
|-----------|-------------|
| **No universal superior model** | LR outperforms GB despite popularity |
| **Algorithms make different assumptions** | NN assumes non-linearity; LR assumes linearity |
| **Problem-specific optimization matters** | Simple model best for linearly-separable data |
| **Must test empirically** | Grid/Random search revealed optimal configurations |

### 🎓 Conclusions

1. ✓ **Split first, then fit** — eliminates data leakage from scaler and PCA
2. ✓ **HVG on train only** — gene selection does not use test variance
3. ✓ **Pipeline for CV and GridSearch** — preprocessing re-fits cleanly in every fold
4. ⚠️ High performance warrants external validation before clinical deployment